# Case 4 Report

Функциональная регрессия через линейные функционалы.

Проверяем:
- сравнение базисов;
- влияние числа функционалов `m`;
- влияние регуляризации `lambda`;
- устойчивость к шуму и изменению сетки.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from case_4.data import SyntheticConfig, create_uniform_grid
from case_4.basis import piecewise_indicator_basis, trigonometric_basis
from case_4.experiments import (
    compare_bases,
    grid_stability_study,
    noise_stability_study,
    sweep_functionals,
    sweep_lambda,
)


In [ ]:
config = SyntheticConfig(n_samples=320, n_grid=200, seed=42)
results = compare_bases(config=config, lambda_value=0.01)
[(r.basis_name, r.metrics.test_rmse, r.metrics.test_r2) for r in results]


In [ ]:
m_points = sweep_functionals([2, 3, 4, 5, 6, 7], config=config, lambda_value=0.01)
lam_points = sweep_lambda([0.0, 1e-4, 1e-3, 1e-2, 1e-1], config=config, harmonics=4)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot([p.value for p in m_points], [p.test_rmse for p in m_points], marker='o')
plt.xlabel('m')
plt.ylabel('test RMSE')
plt.title('RMSE от числа функционалов')

plt.subplot(1, 2, 2)
plt.plot([p.value for p in lam_points], [p.test_rmse for p in lam_points], marker='o')
plt.xscale('log')
plt.xlabel('lambda')
plt.ylabel('test RMSE')
plt.title('RMSE от регуляризации')
plt.tight_layout()


In [ ]:
noise_points = noise_stability_study([0.01, 0.03, 0.05, 0.08], n_grid=200)
grid_points = grid_stability_study([50, 80, 120, 200])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot([p.value for p in noise_points], [p.test_rmse for p in noise_points], marker='o')
plt.xlabel('noise level')
plt.ylabel('test RMSE')
plt.title('Устойчивость к шуму')

plt.subplot(1, 2, 2)
plt.plot([p.value for p in grid_points], [p.test_rmse for p in grid_points], marker='o')
plt.xlabel('grid size')
plt.ylabel('test RMSE')
plt.title('Устойчивость к сетке')
plt.tight_layout()


In [ ]:
t = create_uniform_grid(config.n_grid)
tri = [r for r in results if r.basis_name == 'trigonometric'][0]
pie = [r for r in results if r.basis_name == 'piecewise'][0]

plt.figure(figsize=(8, 4))
plt.plot(t, tri.w_t, label='w(t): trigonometric')
plt.plot(t, pie.w_t, label='w(t): piecewise')
plt.xlabel('t')
plt.ylabel('w(t)')
plt.title('Сравнение восстановленной весовой функции')
plt.legend()
